In [9]:
# input
metaldb_pred = "../../../data/pred_ge_3_clique_3.tsv"
sampled_pred_dir = "./tmp/pred/"
domain_file = "./tmp/sampled_600k_repId-entryId-domain.tsv"

# output
result_file = "./tmp/sampled_pred_result.tsv"

In [5]:
import pandas as pd
from pathlib import Path

def merge_files(path: str) -> pd.DataFrame:

    p = Path(path)
    if p.is_dir():
        dfs = []
        for f in p.iterdir():
            if f.is_file():
                dfs.append(pd.read_table(f))
        return pd.concat(dfs)

    elif p.is_file():
        return pd.read_table(p)

    else:
        raise ValueError

df_gtp = merge_files(sampled_pred_dir)
df_gtp.rename(columns={"pred": "pred_group_type", "posi": "all_ched_posi"}, inplace=True)

df_p = pd.read_table(metaldb_pred)
df_p.rename(columns={"pred": "pred_proba"}, inplace=True)
df = pd.merge(df_gtp, df_p, on="seq_id")
del df_gtp, df_p

In [6]:
records = []
for _, row in df.iterrows():
    seq_id = row["seq_id"]
    positions = row["posi"].split(",")
    all_ched_positions = row["all_ched_posi"].split(",")
    pred_probas = row["pred_proba"].split(",")
    pred_group_types = row["pred_group_type"].split(",")

    posi_to_group_type = dict(zip(all_ched_positions, pred_group_types))
    metal_group_types = [posi_to_group_type[i] for i in positions]
    
    record = {
        "seq_id": seq_id.split("-")[1] if seq_id.startswith("AFDB") else seq_id,
        "pred_seq_num": ",".join(
            [f"{int(i) + 1}" for i in positions]
        ),  # to seq num
        "proba": ",".join(pred_probas),
        "metal_group_type": ",".join(metal_group_types),
    }
    exclude_keys = {
        "seq_id",
        "posi",
        "all_ched_posi",
        "pred_proba",
        "pred_group_type",
    }
    for k in row.index:
        if k not in exclude_keys:
            record[k] = row[k]
    records.append(record)

    

In [10]:
df = pd.DataFrame(records)
df_domain = pd.read_table(domain_file, header=None, names=["rep_id", "seq_id", "domain"])
df = pd.merge(df, df_domain, on="seq_id")
df.to_csv(result_file, sep="\t", index=None)